---
authors:
- name: Steffen Bollmann
- name: Michèle Masson-Trottier
date: '2026-10-21'
license: MIT
---
# MRI Phase Unwrapping with ROMEO



## Purpose

MRI phase data are inherently **wrapped** — phase values outside the range $[-\pi, \pi]$ "wrap around", creating artificial discontinuities. **Phase unwrapping** recovers the true, continuous phase signal, which is essential for downstream analyses such as quantitative susceptibility mapping (QSM) and field mapping.

This tutorial demonstrates how to:

1. Download a demo dataset with magnitude and phase images
2. Prepare the data for unwrapping
3. Run **ROMEO** (Robust phase unwrapping) to recover continuous phase

:::{admonition} Learning Objectives
:class: tip
By the end of this tutorial you will be able to:
- Explain why MRI phase data need unwrapping
- Use ROMEO for robust, magnitude-guided phase unwrapping
- Interpret the unwrapped phase output
:::


## Citation and Resources

### Tools used in this workflow

__ROMEO__
: Dymerska, B., Eckstein, K., et al. (2021). Phase unwrapping with a rapid opensource minimum spanning tree algorithm (ROMEO). *Magnetic Resonance in Medicine*, 85(4), 2294–2308. [https://doi.org/10.1002/mrm.28563](https://doi.org/10.1002/mrm.28563)

### Educational resources

- [ROMEO on GitHub](https://github.com/korbinian90/ROMEO.jl)
- [Neurodesk documentation](https://neurodesk.org)


## Prerequisites

:::{admonition} Before you begin
:class: warning
Make sure you have access to a running Neurodesk instance. See [Getting Set Up with Neurodesk](https://neurodesk.org/getting-started/) for instructions.
:::

- A running Neurodesktop VNC session
- Familiarity with entering commands in a terminal
- [ ] No additional installation required — ROMEO is pre-installed in Neurodesk


## Load software tools and download demo data

We use the same demo dataset as the [SWI tutorial](swi.ipynb). Open a terminal in Neurodesktop and run the following commands to download the data, then copy and prepare the magnitude and phase files for ROMEO.

:::{tip}
To paste in Neurodesktop, best to use the right click on your mouse!
:::

![The terminal in Neurodesktop](https://raw.githubusercontent.com/neurodesk/media/main/screenshots/neurodesk/neurodesktop-terminal.png)

*The terminal in Neurodesktop.*


```bash
mkdir -p ~/neurodesktop-storage/swi-demo/
curl -fL --retry 5 --retry-all-errors -o ~/neurodesktop-storage/swi-demo/01_bids.zip \
   https://huggingface.co/datasets/neurodeskorg/qsm-testdata/resolve/c97dfec3f16c6f4b5570e260a323931ed48db9ef/01_bids.zip

unzip -o ~/neurodesktop-storage/swi-demo/01_bids.zip -d ~/neurodesktop-storage/swi-demo/
```

Now copy the magnitude and phase files into a dedicated working directory:

```bash
mkdir -p ~/neurodesktop-storage/romeo-demo/

cp ~/neurodesktop-storage/swi-demo/01_bids/sub-170705134431std1312211075243167001/ses-1/anat/sub-170705134431std1312211075243167001_ses-1_run-1_part-phase_T2starw.nii \
   ~/neurodesktop-storage/romeo-demo/phase.nii

cp ~/neurodesktop-storage/swi-demo/01_bids/sub-170705134431std1312211075243167001/ses-1/anat/sub-170705134431std1312211075243167001_ses-1_run-1_part-mag_T2starw.nii \
   ~/neurodesktop-storage/romeo-demo/mag.nii
```

:::{note}
The files are already uncompressed `.nii` format, which is what ROMEO expects. No need to gunzip.
:::

![Magnitude and phase inputs ready for ROMEO](/static/tutorials/phase_processing/unwrapping/romeo-inputs.jpg)

*Both copied input files are approximately 16 MB.*


## Run ROMEO for phase unwrapping

Open **Neurodesk → Phase Processing → romeo → romeo 3.2.8**, then run the following command in the new ROMEO container terminal.

![Launch ROMEO](https://raw.githubusercontent.com/neurodesk/media/neurodesk-shared-gui-controls-2026-10-20/screenshots/neurodesk/launch-romeo-guide.png)

:::{note}
The `-k nomask` flag disables brain masking so all voxels are unwrapped. For most applications you would want to use a brain mask — see the [ROMEO documentation](https://github.com/korbinian90/ROMEO.jl) for options.
:::


```bash
romeo \
  -p ~/neurodesktop-storage/romeo-demo/phase.nii \
  -m ~/neurodesktop-storage/romeo-demo/mag.nii \
  -k nomask \
  -o ~/neurodesktop-storage/romeo-demo/
```

![Terminal output after running ROMEO](/static/tutorials/phase_processing/unwrapping/romeo-completed.jpg)

*The command returns to the prompt. `unwrapped.nii` and `settings_romeo.txt` confirm that the output was written.*


## Inspect the results

Verify the output in the ROMEO terminal:

```bash
ls ~/neurodesktop-storage/romeo-demo/
```

1. Open **Neurodesk → Visualization → itksnap → itksnapGUI 4.4.0**. Choose **No** if asked whether to change the radiological layout preferences.

![Launch ITK-SNAP GUI](https://raw.githubusercontent.com/neurodesk/media/neurodesk-shared-gui-controls-2026-10-20/screenshots/neurodesk/launch-itksnap-guide.png)

2. Choose **File → Open Main Image…**, enter `/home/jovyan/neurodesktop-storage/romeo-demo/phase.nii`, then click **Next** and **Finish**.
3. Choose **File → Add Another Image…**, enter `/home/jovyan/neurodesktop-storage/romeo-demo/unwrapped.nii`, then click **Next** and **Finish**.

![Add the unwrapped phase as another image](/static/tutorials/phase_processing/unwrapping/add-comparison-image-annotated.svg)

4. Click the small tiled-layout button beside a slice view to compare both layers. Choose **Tools → Image Contrast → Auto-Adjust Contrast (all layers)**.
5. With `-k nomask`, noisy background values can make the unwrapped brain look dark. Open **Tools → Layer Inspector…**, select **unwrapped** and the **Contrast** tab. For this demo, set **Minimum** to `-10` and **Maximum** to `10`, pressing **Enter** after each value, then **Close**. These settings change the display only.

![Set a useful display range for the unwrapped image](/static/tutorials/phase_processing/unwrapping/phase-contrast-window-annotated.svg)

![ROMEO unwrapping result: wrapped and unwrapped phase side by side](/static/tutorials/phase_processing/unwrapping/romeo-result-annotated.svg)

Each panel shows wrapped phase on the left and unwrapped phase on the right. The sharp phase jumps disappear within the brain; background noise remains because this example deliberately disables masking.


## Summary

In this tutorial you:

1. Downloaded and prepared magnitude and phase MRI data
2. Used ROMEO to perform robust phase unwrapping
3. Compared the wrapped and unwrapped phase images

:::{seealso}
- [SWI tutorial](swi.ipynb) — for susceptibility-weighted imaging using the same dataset
- [QSMxT example](../../examples/quantitative_imaging/qsmxt.ipynb) — for quantitative susceptibility mapping with QSMxT
:::